# UTD-MHAD: Comprehensive Per-Fold Feature Selection Comparison

This notebook compares **17 feature selection methods** across all 8 LOSO folds (treated individually):

**Standard Methods:**
1. Mutual Information (filter)
2. RFE (wrapper)
3. LASSO/L1 (embedded)

**14 EvoloPy Metaheuristics:**
BAT, CS, DE, FFA, GA, GWO, HHO, JAYA, MFO, MVO, PSO, SCA, SSA, WOA

All results are saved **per fold** (no averaging) inside `results_per_fold/`.
Comprehensive plots and CSVs are saved inside `results_per_fold/plots/`.


In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [2]:
import torch
print(torch.cuda.device_count())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

1
NVIDIA GeForce RTX 4090


In [3]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import time
import warnings
from pathlib import Path
from tqdm import tqdm
from copy import deepcopy
import random
import sys
from scipy import stats
import psutil
import tracemalloc
import gc
import json as json_lib

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                             precision_score, recall_score, classification_report)

# Standard feature selection imports
from sklearn.feature_selection import mutual_info_classif, RFE, SelectKBest
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

# Import ALL EvoloPy optimizers
sys.path.append('../EvoloPy-master')
from EvoloPy.optimizers import BAT, CS, DE, FFA, GA, GWO, HHO, JAYA, MFO, MVO, PSO, SCA, SSA, WOA

warnings.filterwarnings('ignore')
np.random.seed(42)
torch.manual_seed(42)
random.seed(42)

# Global device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"PyTorch version: {torch.__version__}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")


Device: cuda
PyTorch version: 2.9.1+cu128
GPU: NVIDIA GeForce RTX 4090


## Configuration

In [4]:
# Paths
FEATURE_DIR = Path("features_new")

# Master output directory
RESULTS_ROOT = Path("results_new_no_video")
RESULTS_ROOT.mkdir(exist_ok=True)
PLOTS_DIR = RESULTS_ROOT / "plots"
PLOTS_DIR.mkdir(exist_ok=True)

# Hyperparameters
N_FOLDS = 8  # LOSO
N_EPOCHS = 300
BATCH_SIZE = 32
LEARNING_RATE = 0.001
HIDDEN_DIM = 128

# Feature selection hyperparameters - metaheuristic
N_POPULATION = 20
MAX_ITERATIONS = 30

# For standard methods (target: select ~50% of features)
TARGET_FEATURE_PERCENTAGE = 0.5

# All EvoloPy optimizer names and their callable entry points
EVOLOPY_OPTIMIZERS = {
    'BAT': BAT.BAT,
    'CS':  CS.CS,
    'DE':  DE.DE,
    'FFA': FFA.FFA,
    'GA':  GA.GA,
    'GWO': GWO.GWO,
    'HHO': HHO.HHO,
    'JAYA': JAYA.JAYA,
    'MFO': MFO.MFO,
    'MVO': MVO.MVO,
    'PSO': PSO.PSO,
    'SCA': SCA.SCA,
    'SSA': SSA.SSA,
    'WOA': WOA.WOA,
}

ALL_METHODS = ['baseline', 'mutual_info', 'rfe', 'lasso'] + [f'meta_{k}' for k in EVOLOPY_OPTIMIZERS.keys()]

# Create per-method subdirectories
for method in ALL_METHODS:
    (RESULTS_ROOT / method).mkdir(exist_ok=True)

print(f"Configuration:")
print(f"  N_FOLDS: {N_FOLDS}")
print(f"  N_EPOCHS: {N_EPOCHS}")
print(f"  Target feature selection: {TARGET_FEATURE_PERCENTAGE*100:.0f}%")
print(f"  Metaheuristic pop: {N_POPULATION}, iter: {MAX_ITERATIONS}")
print(f"  EvoloPy optimizers: {list(EVOLOPY_OPTIMIZERS.keys())}")
print(f"  Total methods (incl. baseline): {len(ALL_METHODS)}")


Configuration:
  N_FOLDS: 8
  N_EPOCHS: 300
  Target feature selection: 50%
  Metaheuristic pop: 20, iter: 30
  EvoloPy optimizers: ['BAT', 'CS', 'DE', 'FFA', 'GA', 'GWO', 'HHO', 'JAYA', 'MFO', 'MVO', 'PSO', 'SCA', 'SSA', 'WOA']
  Total methods (incl. baseline): 18


## Load Data

In [5]:
# Load features
X_feat = joblib.load(FEATURE_DIR / "X_feat.pkl")
y = np.load(FEATURE_DIR / "y.npy")
subjects = np.load(FEATURE_DIR / "subjects.npy")
le = joblib.load(FEATURE_DIR / "label_encoder.pkl")
print(f"Loaded {len(X_feat)} samples")
print(f"Number of classes: {len(np.unique(y))}")
print(f"Number of subjects: {len(np.unique(subjects))}")
# Get feature dimensions
first_sample = X_feat[0]
FEATURE_DIMS = {
'depth': first_sample['depth_feat'].shape[0],
'sensor': first_sample['sensor_feat'].shape[0],
'skeleton': first_sample['skeleton_feat'].shape[0]
}
TOTAL_FEATURES = sum(FEATURE_DIMS.values())
print(f"\nFeature dimensions:")
for modality, dim in FEATURE_DIMS.items():
    print(f"  {modality}: {dim}")
print(f"  TOTAL: {TOTAL_FEATURES}")


Loaded 861 samples
Number of classes: 27
Number of subjects: 8

Feature dimensions:
  depth: 216
  sensor: 652
  skeleton: 1645
  TOTAL: 2513


## Neural Network Model (Same as Original)

In [6]:
class MultiModalDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.FloatTensor(features)
        self.labels = torch.LongTensor(labels)
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]


class SimpleNN(nn.Module):
    """MLP for unified feature vector (adaptive to feature subset size)"""
    def __init__(self, input_dim, num_classes):
        super().__init__()
        
        # Adaptive hidden layer sizing based on input dimension
        hidden1 = max(128, min(512, input_dim * 2))
        hidden2 = max(64, min(256, hidden1 // 2))
        
        self.classifier = nn.Sequential(
            nn.Linear(input_dim, hidden1),
            nn.BatchNorm1d(hidden1),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(hidden1, hidden2),
            nn.BatchNorm1d(hidden2),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden2, num_classes)
        )
    
    def forward(self, x):
        return self.classifier(x)

print("Neural network model defined")


Neural network model defined


## Helper Functions

In [7]:
def prepare_unified_features(X_feat_list, feature_mask=None):
    """Concatenate all modality features into unified vector"""
    unified_features = []
    
    for sample in X_feat_list:
        feat_vector = np.concatenate([
            sample['depth_feat'],
            sample['sensor_feat'],
            sample['skeleton_feat']
        ])
        
        if feature_mask is not None:
            feat_vector = feat_vector[feature_mask]
        
        unified_features.append(feat_vector)
    
    return np.array(unified_features)

def calculate_modality_retention(binary_mask, feature_dims):
    """Calculate how many features retained per modality"""
    start_idx = 0
    retention = {}
    
    for modality, dim in feature_dims.items():
        end_idx = start_idx + dim
        modality_mask = binary_mask[start_idx:end_idx]
        num_selected = np.sum(modality_mask)
        percentage = (num_selected / dim) * 100
        
        retention[modality] = {
            'selected': int(num_selected),
            'total': dim,
            'percentage': percentage
        }
        start_idx = end_idx
    
    return retention

def train_and_evaluate(model, train_loader, val_loader, test_loader, num_epochs, lr):
    """Train model and return metrics"""
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    model.train()
    for epoch in range(num_epochs):
        for features, labels in train_loader:
            features = features.to(DEVICE)
            labels = labels.to(DEVICE)
            
            optimizer.zero_grad()
            outputs = model(features)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
    
    # Evaluate
    model.eval()
    with torch.no_grad():
        # Validation
        val_preds, val_true = [], []
        for features, labels in val_loader:
            features = features.to(DEVICE)
            outputs = model(features)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            val_preds.extend(preds)
            val_true.extend(labels.numpy())
        val_acc = accuracy_score(val_true, val_preds)
        
        # Test
        test_preds, test_true = [], []
        for features, labels in test_loader:
            features = features.to(DEVICE)
            outputs = model(features)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            test_preds.extend(preds)
            test_true.extend(labels.numpy())
        test_acc = accuracy_score(test_true, test_preds)
    
    return val_acc, test_acc

def count_model_parameters(model):
    """Count trainable parameters in a model"""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def get_model_size_mb(model):
    """Get model size in MB"""
    param_size = sum(p.nelement() * p.element_size() for p in model.parameters())
    buffer_size = sum(b.nelement() * b.element_size() for b in model.buffers())
    return (param_size + buffer_size) / (1024 ** 2)

def get_gpu_memory_mb():
    """Get current GPU memory usage in MB"""
    if torch.cuda.is_available():
        return torch.cuda.memory_allocated() / (1024 ** 2)
    return 0.0

def get_dataset_size_mb(X):
    """Get dataset size in MB"""
    return X.nbytes / (1024 ** 2)

print("Helper functions defined")


Helper functions defined


## Enhanced Evaluation (returns full metrics per fold)

In [8]:
def train_and_evaluate_full(model, train_loader, val_loader, test_loader, num_epochs, lr, num_classes):
    """Train model and return comprehensive metrics including predictions for confusion matrix"""
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    # Track GPU memory before training
    gpu_mem_before = get_gpu_memory_mb()
    train_start = time.time()

    train_losses = []
    best_val_acc = -1.0
    best_model_state = None
    
    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0.0

        for features, labels in train_loader:
            features = features.to(DEVICE)
            labels = labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(features)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        train_losses.append(epoch_loss / len(train_loader))

        # Check validation accuracy after each epoch
        model.eval()
        val_correct, val_total = 0, 0
        with torch.no_grad():
            for features, labels in val_loader:
                features = features.to(DEVICE)
                outputs = model(features)
                preds = torch.argmax(outputs, dim=1)
                val_correct += (preds.cpu() == labels).sum().item()
                val_total += labels.size(0)

        epoch_val_acc = val_correct / val_total

        if epoch_val_acc > best_val_acc:
            best_val_acc = epoch_val_acc
            best_model_state = deepcopy(model.state_dict())
    
    train_time = time.time() - train_start
    gpu_mem_after = get_gpu_memory_mb()

    # Restore best model
    model.load_state_dict(best_model_state)
    
    # Final evaluation
    model.eval()
    with torch.no_grad():
        val_preds, val_true = [], []
        for features, labels in val_loader:
            features = features.to(DEVICE)
            outputs = model(features)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            val_preds.extend(preds)
            val_true.extend(labels.numpy())
        
        # Test
        test_preds, test_true = [], []
        for features, labels in test_loader:
            features = features.to(DEVICE)
            outputs = model(features)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            test_preds.extend(preds)
            test_true.extend(labels.numpy())
    
    val_acc = accuracy_score(val_true, val_preds)
    test_acc = accuracy_score(test_true, test_preds)
    
    metrics = {
        'val_acc': val_acc,
        'test_acc': test_acc,
        'test_f1_macro': f1_score(test_true, test_preds, average='macro', zero_division=0),
        'test_f1_weighted': f1_score(test_true, test_preds, average='weighted', zero_division=0),
        'test_precision_macro': precision_score(test_true, test_preds, average='macro', zero_division=0),
        'test_recall_macro': recall_score(test_true, test_preds, average='macro', zero_division=0),
        'test_preds': np.array(test_preds),
        'test_true': np.array(test_true),
        'val_preds': np.array(val_preds),
        'val_true': np.array(val_true),
        'train_time_sec': train_time,
        'gpu_mem_before_mb': gpu_mem_before,
        'gpu_mem_after_mb': gpu_mem_after,
        'gpu_mem_peak_mb': torch.cuda.max_memory_allocated() / (1024**2) if torch.cuda.is_available() else 0,
        'model_params': count_model_parameters(model),
        'model_size_mb': get_model_size_mb(model),
        'train_losses': train_losses,
    }
    return metrics

print("Enhanced evaluation function defined")


Enhanced evaluation function defined


## Feature Selection Methods\n### 0. EvoloPy Metaheuristics

In [9]:
# ============================================================================
# EXACT EVOLOPY BAT IMPLEMENTATION FROM ORIGINAL CODE
# ============================================================================

def train_model_quick(model, train_loader, val_loader, epochs, lr, device):
    """Quick training for fitness evaluation"""
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    best_val_loss = float('inf')
    best_val_acc = 0.0
    
    for epoch in range(epochs):
        model.train()
        for features, labels in train_loader:
            features, labels = features.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(features), labels)
            loss.backward()
            optimizer.step()
        
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for features, labels in val_loader:
                features, labels = features.to(device), labels.to(device)
                outputs = model(features)
                val_loss += criterion(outputs, labels).item()
                val_correct += (torch.argmax(outputs, 1) == labels).sum().item()
                val_total += labels.size(0)
        
        val_loss /= len(val_loader)
        val_acc = val_correct / val_total
        if val_loss < best_val_loss:
            best_val_loss, best_val_acc = val_loss, val_acc
    
    return best_val_loss, best_val_acc

# ============================================================================
# FITNESS FUNCTION: Using (1 - accuracy)
# ============================================================================
# Rationale: (1 - accuracy) is preferred over loss because:
#   - It directly optimizes the metric we care about (accuracy)
#   - It is bounded in [0, 1], making it cleaner for metaheuristic optimization
#   - Loss can vary in scale across different feature subsets
#   - Accuracy-based fitness is more interpretable and comparable across methods
#   - Less sensitive to calibration issues of the neural network

def create_fitness_function_evolopy(X_train, y_train, X_val, y_val, num_classes):
    """Create fitness function for EvoloPy using (1 - accuracy) + feature penalty"""
    eval_count = [0]  # track evaluations
    
    def fitness_function(binary_mask):
        try:
            if binary_mask.dtype != bool:
                binary_mask = binary_mask > 0.5

            num_selected = np.sum(binary_mask)
            if num_selected == 0:
                return 1.0
            
            X_tr_sel = X_train[:, binary_mask]
            X_val_sel = X_val[:, binary_mask]
            
            train_dataset = MultiModalDataset(X_tr_sel, y_train)
            val_dataset = MultiModalDataset(X_val_sel, y_val)
            train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
            val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
            
            model = SimpleNN(X_tr_sel.shape[1], num_classes).to(DEVICE)
            
            val_loss, val_acc = train_model_quick(
                model, train_loader, val_loader,
                epochs=30, lr=1e-3, device=DEVICE
            )
            
            del model, train_dataset, val_dataset, train_loader, val_loader
            torch.cuda.empty_cache()
            
            eval_count[0] += 1
            
            # Weighted fitness: accuracy + feature reduction pressure
            alpha = 0.95  # weight for accuracy
            beta = 0.05   # weight for feature reduction
            feature_ratio = num_selected / len(binary_mask)

            fitness = alpha * (1.0 - val_acc) + beta * feature_ratio
            return fitness
            
        except Exception as e:
            print(f"Error in fitness: {e}")
            return 1.0
        
    return fitness_function


def s_transfer(x):
    return 1 / (1 + np.exp(-10 * (x - 0.5)))


def run_evolopy_optimizer(optimizer_name, optimizer_func, X_train, y_train, X_val, y_val, num_classes):
    """Run any EvoloPy optimizer generically"""
    print(f"    Running EvoloPy {optimizer_name}...")
    print(f"      Fitness: (1 - accuracy), Pop: {N_POPULATION}, Iter: {MAX_ITERATIONS}")
    
    fitness_func = create_fitness_function_evolopy(X_train, y_train, X_val, y_val, num_classes)
    
    start = time.time()
    solution = optimizer_func(fitness_func, 0, 1, TOTAL_FEATURES, N_POPULATION, MAX_ITERATIONS)
    exec_time = time.time() - start
    
    # binary_mask = solution.bestIndividual > 0.5
    binary_mask = np.random.rand(TOTAL_FEATURES) < s_transfer(solution.bestIndividual)
    modality_ret = calculate_modality_retention(binary_mask, FEATURE_DIMS)
    
    results = {
        'mask': binary_mask,
        'convergence': solution.convergence.tolist() if hasattr(solution.convergence, 'tolist') else list(solution.convergence),
        'best_fitness': float(solution.convergence[-1]) if len(solution.convergence) > 0 else float('inf'),
        'execution_time': exec_time,
        'num_selected': int(np.sum(binary_mask)),
        'num_total': len(binary_mask),
        'modality_retention': modality_ret,
        'method': f'meta_{optimizer_name}'
    }
    
    print(f"      Selected: {results['num_selected']}/{results['num_total']} "
          f"({results['num_selected']/results['num_total']*100:.1f}%), "
          f"Fitness: {results['best_fitness']:.4f}, Time: {exec_time:.1f}s")
    print(f"      Modality: D={modality_ret['depth']['percentage']:.0f}%, "
          f"S={modality_ret['sensor']['percentage']:.0f}%, "
          f"Sk={modality_ret['skeleton']['percentage']:.0f}%")
    
    return results

print("All EvoloPy metaheuristic runners defined")
print(f"Available optimizers: {list(EVOLOPY_OPTIMIZERS.keys())}")


All EvoloPy metaheuristic runners defined
Available optimizers: ['BAT', 'CS', 'DE', 'FFA', 'GA', 'GWO', 'HHO', 'JAYA', 'MFO', 'MVO', 'PSO', 'SCA', 'SSA', 'WOA']


### 1-3. Standard Feature Selection Methods (Mutual Info, RFE, LASSO)

In [10]:
def run_mutual_info_fs(X_train, y_train, X_val, y_val, num_classes):
    """Run Mutual Information feature selection"""
    print("    Running Mutual Information...")
    
    start_time = time.time()
    
    # Calculate mutual information scores
    mi_scores = mutual_info_classif(X_train, y_train, random_state=42)
    
    # Select top k features (target percentage)
    k = int(TOTAL_FEATURES * TARGET_FEATURE_PERCENTAGE)
    top_k_indices = np.argsort(mi_scores)[::-1][:k]
    
    binary_mask = np.zeros(TOTAL_FEATURES, dtype=bool)
    binary_mask[top_k_indices] = True
    
    execution_time = time.time() - start_time
    
    modality_retention = calculate_modality_retention(binary_mask, FEATURE_DIMS)
    
    results = {
        'mask': binary_mask,
        'execution_time': execution_time,
        'num_selected': int(np.sum(binary_mask)),
        'num_total': len(binary_mask),
        'modality_retention': modality_retention,
        'mi_scores': mi_scores,
        'method': 'mutual_info'
    }
    
    print(f"      Selected: {results['num_selected']}/{results['num_total']} "
          f"({results['num_selected']/results['num_total']*100:.1f}%), "
          f"Time: {results['execution_time']:.1f}s")
    
    return results


def run_rfe_fs(X_train, y_train, X_val, y_val, num_classes):
    """Run RFE feature selection"""
    print("    Running RFE...")
    
    start_time = time.time()
    
    # Use Random Forest as base estimator
    estimator = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)
    
    # Select top k features
    k = int(TOTAL_FEATURES * TARGET_FEATURE_PERCENTAGE)
    selector = RFE(estimator, n_features_to_select=k, step=50)  # Remove 50 features at a time
    selector.fit(X_train, y_train)
    
    binary_mask = selector.support_
    
    execution_time = time.time() - start_time
    
    modality_retention = calculate_modality_retention(binary_mask, FEATURE_DIMS)
    
    results = {
        'mask': binary_mask,
        'execution_time': execution_time,
        'num_selected': int(np.sum(binary_mask)),
        'num_total': len(binary_mask),
        'modality_retention': modality_retention,
        'ranking': selector.ranking_,
        'method': 'rfe'
    }
    
    print(f"      Selected: {results['num_selected']}/{results['num_total']} "
          f"({results['num_selected']/results['num_total']*100:.1f}%), "
          f"Time: {results['execution_time']:.1f}s")
    
    return results


def run_lasso_fs(X_train, y_train, X_val, y_val, num_classes):
    """Run LASSO feature selection"""
    print("    Running LASSO...")
    start_time = time.time()

    # Train Lasso to get feature importance scores
    lasso = LogisticRegression(
        penalty='l1',
        C=0.01,
        solver='liblinear',
        random_state=42,
        max_iter=1000
    )
    lasso.fit(X_train, y_train)

    # Rank features by coefficient magnitude across all classes
    coef_abs = np.abs(lasso.coef_).sum(axis=0)

    # Select top-k features to match target percentage
    target_count = int(TOTAL_FEATURES * TARGET_FEATURE_PERCENTAGE)
    top_indices = np.argsort(coef_abs)[::-1][:target_count]
    binary_mask = np.zeros(TOTAL_FEATURES, dtype=bool)
    binary_mask[top_indices] = True

    execution_time = time.time() - start_time
    modality_retention = calculate_modality_retention(binary_mask, FEATURE_DIMS)

    results = {
        'mask': binary_mask,
        'execution_time': execution_time,
        'num_selected': int(np.sum(binary_mask)),
        'num_total': len(binary_mask),
        'modality_retention': modality_retention,
        'coefficients': coef_abs,
        'method': 'lasso'
    }

    print(f"      Selected: {results['num_selected']}/{results['num_total']} "
          f"({results['num_selected']/results['num_total']*100:.1f}%), "
          f"Time: {results['execution_time']:.1f}s")

    return results

print("Standard FS methods defined (Mutual Info, RFE, LASSO)")


Standard FS methods defined (Mutual Info, RFE, LASSO)


## Main Per-Fold Experiment Runner

In [11]:
def run_all_methods_per_fold():
    """
    Run ALL methods across ALL folds, saving comprehensive per-fold results.
    Each fold is treated as an independent dataset version (no averaging).
    """
    print("="*80)
    print("STARTING COMPREHENSIVE PER-FOLD EXPERIMENTS")
    print(f"Methods: baseline + 3 standard + {len(EVOLOPY_OPTIMIZERS)} metaheuristics = {len(ALL_METHODS)} total")
    print(f"Folds: {N_FOLDS}")
    print("="*80)
    
    # Prepare unified features
    X_unified = prepare_unified_features(X_feat)
    num_classes = len(np.unique(y))
    
    # Master results dict: {method_name: {fold_idx: {...}}}
    master_results = {m: {} for m in ALL_METHODS}
    
    # LOSO cross-validation
    gkf = GroupKFold(n_splits=N_FOLDS)
    
    for fold_idx, (train_val_idx, test_idx) in enumerate(gkf.split(X_unified, y, groups=subjects)):
        print(f"\n{'#'*80}")
        print(f"# FOLD {fold_idx + 1}/{N_FOLDS}")
        print(f"{'#'*80}")
        
        # Split data
        X_train_val = X_unified[train_val_idx]
        y_train_val = y[train_val_idx]
        X_test = X_unified[test_idx]
        y_test = y[test_idx]
        
        # Further split train_val into train and val
        n_val = len(train_val_idx) // 5
        val_indices = np.random.choice(len(train_val_idx), n_val, replace=False)
        train_indices = np.array([i for i in range(len(train_val_idx)) if i not in val_indices])
        
        X_train = X_train_val[train_indices]
        y_train = y_train_val[train_indices]
        X_val = X_train_val[val_indices]
        y_val = y_train_val[val_indices]
        
        print(f"  Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")
        
        # Normalize
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_val = scaler.transform(X_val)
        X_test = scaler.transform(X_test)
        
        # Original dataset size
        orig_dataset_size_mb = get_dataset_size_mb(X_train) + get_dataset_size_mb(X_val) + get_dataset_size_mb(X_test)
        
        # =====================================================================
        # Run each method
        # =====================================================================
        for method_name in ALL_METHODS:
            print(f"\n  --- {method_name.upper()} ---")
            
            fs_start_time = time.time()
            
            # Feature selection
            if method_name == 'baseline':
                feature_mask = np.ones(TOTAL_FEATURES, dtype=bool)
                fs_results = {
                    'mask': feature_mask,
                    'execution_time': 0,
                    'num_selected': TOTAL_FEATURES,
                    'num_total': TOTAL_FEATURES,
                    'method': 'baseline'
                }
            elif method_name == 'mutual_info':
                fs_results = run_mutual_info_fs(X_train, y_train, X_val, y_val, num_classes)
                feature_mask = fs_results['mask']
            elif method_name == 'rfe':
                fs_results = run_rfe_fs(X_train, y_train, X_val, y_val, num_classes)
                feature_mask = fs_results['mask']
            elif method_name == 'lasso':
                fs_results = run_lasso_fs(X_train, y_train, X_val, y_val, num_classes)
                feature_mask = fs_results['mask']
            elif method_name.startswith('meta_'):
                opt_name = method_name.replace('meta_', '')
                opt_func = EVOLOPY_OPTIMIZERS[opt_name]
                fs_results = run_evolopy_optimizer(opt_name, opt_func, X_train, y_train, X_val, y_val, num_classes)
                feature_mask = fs_results['mask']
            else:
                raise ValueError(f"Unknown method: {method_name}")
            
            fs_time = time.time() - fs_start_time
            
            # Apply feature mask
            X_train_sel = X_train[:, feature_mask]
            X_val_sel = X_val[:, feature_mask]
            X_test_sel = X_test[:, feature_mask]
            
            # Dataset size after selection
            opt_dataset_size_mb = get_dataset_size_mb(X_train_sel) + get_dataset_size_mb(X_val_sel) + get_dataset_size_mb(X_test_sel)
            
            # Build and train final model
            model = SimpleNN(X_train_sel.shape[1], num_classes).to(DEVICE)
            
            train_dataset = MultiModalDataset(X_train_sel, y_train)
            val_dataset = MultiModalDataset(X_val_sel, y_val)
            test_dataset = MultiModalDataset(X_test_sel, y_test)
            
            train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
            val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
            test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)
            
            if torch.cuda.is_available():
                torch.cuda.reset_peak_memory_stats()
            
            eval_metrics = train_and_evaluate_full(
                model, train_loader, val_loader, test_loader, N_EPOCHS, LEARNING_RATE, num_classes
            )
            
            # Modality retention
            if method_name != 'baseline':
                modality_ret = fs_results.get('modality_retention', calculate_modality_retention(feature_mask, FEATURE_DIMS))
            else:
                modality_ret = {m: {'selected': d, 'total': d, 'percentage': 100.0} for m, d in FEATURE_DIMS.items()}
            
            # Compile comprehensive fold result
            fold_result = {
                'fold': fold_idx,
                'method': method_name,
                'num_train': len(X_train),
                'num_val': len(X_val),
                'num_test': len(X_test),
                # Accuracy & classification metrics
                'val_acc': eval_metrics['val_acc'],
                'test_acc': eval_metrics['test_acc'],
                'test_f1_macro': eval_metrics['test_f1_macro'],
                'test_f1_weighted': eval_metrics['test_f1_weighted'],
                'test_precision_macro': eval_metrics['test_precision_macro'],
                'test_recall_macro': eval_metrics['test_recall_macro'],
                # Feature selection info
                'num_features_selected': int(np.sum(feature_mask)),
                'num_features_total': TOTAL_FEATURES,
                'feature_retention_pct': float(np.sum(feature_mask) / TOTAL_FEATURES * 100),
                'modality_retention': modality_ret,
                'feature_mask': feature_mask,
                # Timing
                'fs_execution_time': fs_results.get('execution_time', 0),
                'train_time_sec': eval_metrics['train_time_sec'],
                'total_time_sec': fs_time + eval_metrics['train_time_sec'],
                # Model size
                'model_params': eval_metrics['model_params'],
                'model_size_mb': eval_metrics['model_size_mb'],
                # Dataset size
                'original_dataset_size_mb': orig_dataset_size_mb,
                'optimized_dataset_size_mb': opt_dataset_size_mb,
                'dataset_reduction_pct': float((1 - opt_dataset_size_mb / orig_dataset_size_mb) * 100) if orig_dataset_size_mb > 0 else 0,
                # GPU
                'gpu_mem_peak_mb': eval_metrics['gpu_mem_peak_mb'],
                # Convergence (metaheuristics only)
                'convergence': fs_results.get('convergence', None),
                'best_fitness': fs_results.get('best_fitness', None),
            }
            
            master_results[method_name][fold_idx] = fold_result
            
            # Save per-fold result immediately
            fold_dir = RESULTS_ROOT / method_name
            fold_save = {k: v for k, v in fold_result.items() if k not in ['feature_mask']}
            # Convert numpy to python types for JSON
            fold_save_clean = {}
            for k, v in fold_save.items():
                if isinstance(v, np.ndarray):
                    fold_save_clean[k] = v.tolist()
                elif isinstance(v, (np.floating, np.integer)):
                    fold_save_clean[k] = float(v)
                elif isinstance(v, dict):
                    fold_save_clean[k] = {}
                    for kk, vv in v.items():
                        if isinstance(vv, dict):
                            fold_save_clean[k][kk] = {kkk: float(vvv) if isinstance(vvv, (np.floating, np.integer)) else vvv for kkk, vvv in vv.items()}
                        else:
                            fold_save_clean[k][kk] = float(vv) if isinstance(vv, (np.floating, np.integer)) else vv
                else:
                    fold_save_clean[k] = v
            
            with open(fold_dir / f"fold_{fold_idx+1}.json", 'w') as f:
                json_lib.dump(fold_save_clean, f, indent=2, default=str)
            
            np.save(fold_dir / f"fold_{fold_idx+1}_mask.npy", feature_mask)
            
            print(f"    Test Acc: {eval_metrics['test_acc']*100:.2f}%, "
                  f"F1: {eval_metrics['test_f1_macro']*100:.2f}%, "
                  f"Features: {np.sum(feature_mask)}/{TOTAL_FEATURES} "
                  f"({np.sum(feature_mask)/TOTAL_FEATURES*100:.1f}%), "
                  f"Model: {eval_metrics['model_size_mb']:.3f}MB")
            
            # Cleanup
            del model, train_dataset, val_dataset, test_dataset
            del train_loader, val_loader, test_loader
            torch.cuda.empty_cache()
            gc.collect()
    
    print(f"\n{'='*80}")
    print("ALL PER-FOLD EXPERIMENTS COMPLETED")
    print(f"{'='*80}")
    
    return master_results

print("Per-fold experiment runner defined")


Per-fold experiment runner defined


## Run All Experiments

In [12]:
master_results = run_all_methods_per_fold()

STARTING COMPREHENSIVE PER-FOLD EXPERIMENTS
Methods: baseline + 3 standard + 14 metaheuristics = 18 total
Folds: 8

################################################################################
# FOLD 1/8
################################################################################
  Train: 603, Val: 150, Test: 108

  --- BASELINE ---
    Test Acc: 97.22%, F1: 97.21%, Features: 2513/2513 (100.0%), Model: 5.449MB

  --- MUTUAL_INFO ---
    Running Mutual Information...
      Selected: 1256/2513 (50.0%), Time: 14.6s
    Test Acc: 99.07%, F1: 99.06%, Features: 1256/2513 (50.0%), Model: 2.994MB

  --- RFE ---
    Running RFE...
      Selected: 1256/2513 (50.0%), Time: 2.6s
    Test Acc: 96.30%, F1: 95.06%, Features: 1256/2513 (50.0%), Model: 2.994MB

  --- LASSO ---
    Running LASSO...
      Selected: 1256/2513 (50.0%), Time: 0.5s
    Test Acc: 87.96%, F1: 86.73%, Features: 1256/2513 (50.0%), Model: 2.994MB

  --- META_BAT ---
    Running EvoloPy BAT...
      Fitness: (1 - accuracy)

## Build Comprehensive Per-Fold Results CSV

In [13]:
# Build a single massive DataFrame with every fold x method combination
rows = []

for method_name in ALL_METHODS:
    for fold_idx in range(N_FOLDS):
        if fold_idx not in master_results[method_name]:
            continue
        r = master_results[method_name][fold_idx]
        
        row = {
            'Method': method_name,
            'Fold': fold_idx + 1,
            'Test Accuracy (%)': r['test_acc'] * 100,
            'Val Accuracy (%)': r['val_acc'] * 100,
            'F1 Macro (%)': r['test_f1_macro'] * 100,
            'F1 Weighted (%)': r['test_f1_weighted'] * 100,
            'Precision Macro (%)': r['test_precision_macro'] * 100,
            'Recall Macro (%)': r['test_recall_macro'] * 100,
            'Features Selected': r['num_features_selected'],
            'Features Total': r['num_features_total'],
            'Feature Retention (%)': r['feature_retention_pct'],
            'FS Time (s)': r['fs_execution_time'],
            'Train Time (s)': r['train_time_sec'],
            'Total Time (s)': r['total_time_sec'],
            'Model Params': r['model_params'],
            'Model Size (MB)': r['model_size_mb'],
            'Original Dataset (MB)': r['original_dataset_size_mb'],
            'Optimized Dataset (MB)': r['optimized_dataset_size_mb'],
            'Dataset Reduction (%)': r['dataset_reduction_pct'],
            'GPU Peak (MB)': r['gpu_mem_peak_mb'],
        }
        
        # Add per-modality retention
        for mod_name in FEATURE_DIMS.keys():
            mod_ret = r['modality_retention'].get(mod_name, {})
            if isinstance(mod_ret, dict):
                row[f'{mod_name}_retention (%)'] = mod_ret.get('percentage', 0)
                row[f'{mod_name}_selected'] = mod_ret.get('selected', 0)
            else:
                row[f'{mod_name}_retention (%)'] = float(mod_ret)
        
        rows.append(row)

df_all = pd.DataFrame(rows)
df_all = df_all.round(4)

# Save
csv_path = RESULTS_ROOT / "all_results_per_fold.csv"
df_all.to_csv(csv_path, index=False)
print(f"Saved comprehensive per-fold CSV: {csv_path}")
print(f"Shape: {df_all.shape}")
print(df_all.head(20).to_string())


Saved comprehensive per-fold CSV: results_new_no_video/all_results_per_fold.csv
Shape: (144, 26)
         Method  Fold  Test Accuracy (%)  Val Accuracy (%)  F1 Macro (%)  F1 Weighted (%)  Precision Macro (%)  Recall Macro (%)  Features Selected  Features Total  Feature Retention (%)  FS Time (s)  Train Time (s)  Total Time (s)  Model Params  Model Size (MB)  Original Dataset (MB)  Optimized Dataset (MB)  Dataset Reduction (%)  GPU Peak (MB)  depth_retention (%)  depth_selected  sensor_retention (%)  sensor_selected  skeleton_retention (%)  skeleton_selected
0      baseline     1            97.2222             100.0       97.2075          97.2075              97.4074           97.2222               2513            2513               100.0000       0.0000         12.5116         12.5117       1426971           5.4493                16.5077                 16.5077                 0.0000        50.2388             100.0000             216              100.0000              652             

## Per-Method Summary Table (with per-fold detail)

In [14]:
# Also create a summary table (mean/std across folds)
summary_rows = []
for method_name in ALL_METHODS:
    method_df = df_all[df_all['Method'] == method_name]
    if len(method_df) == 0:
        continue
    
    row = {
        'Method': method_name,
        'Mean Test Acc (%)': method_df['Test Accuracy (%)'].mean(),
        'Std Test Acc (%)': method_df['Test Accuracy (%)'].std(),
        'Mean F1 Macro (%)': method_df['F1 Macro (%)'].mean(),
        'Std F1 Macro (%)': method_df['F1 Macro (%)'].std(),
        'Mean Features Selected': method_df['Features Selected'].mean(),
        'Mean Feature Retention (%)': method_df['Feature Retention (%)'].mean(),
        'Mean FS Time (s)': method_df['FS Time (s)'].mean(),
        'Mean Train Time (s)': method_df['Train Time (s)'].mean(),
        'Mean Total Time (s)': method_df['Total Time (s)'].mean(),
        'Mean Model Params': method_df['Model Params'].mean(),
        'Mean Model Size (MB)': method_df['Model Size (MB)'].mean(),
        'Mean Dataset Reduction (%)': method_df['Dataset Reduction (%)'].mean(),
        'Mean GPU Peak (MB)': method_df['GPU Peak (MB)'].mean(),
    }
    summary_rows.append(row)

df_summary = pd.DataFrame(summary_rows).round(2)
df_summary.to_csv(RESULTS_ROOT / "summary_all_methods.csv", index=False)
print("\nSUMMARY TABLE (mean across folds):")
print(df_summary.to_string(index=False))



SUMMARY TABLE (mean across folds):
     Method  Mean Test Acc (%)  Std Test Acc (%)  Mean F1 Macro (%)  Std F1 Macro (%)  Mean Features Selected  Mean Feature Retention (%)  Mean FS Time (s)  Mean Train Time (s)  Mean Total Time (s)  Mean Model Params  Mean Model Size (MB)  Mean Dataset Reduction (%)  Mean GPU Peak (MB)
   baseline              94.32              4.65              93.44              5.32                 2513.00                      100.00              0.00                11.86                11.86          1426971.0                  5.45                        0.00               50.24
mutual_info              93.85              4.40              92.59              5.45                 1256.00                       49.98             14.61                10.72                25.33           783387.0                  2.99                       50.02               35.36
        rfe              93.28              5.68              91.86              6.84                 1

## Comprehensive Visualizations

In [15]:
# ============================================================================
# PLOT 1: Test Accuracy per fold for all methods (heatmap)
# ============================================================================
pivot = df_all.pivot_table(values='Test Accuracy (%)', index='Method', columns='Fold')
pivot = pivot.reindex(ALL_METHODS)

fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='YlGnBu', ax=ax, linewidths=0.5, cbar_kws={'label': 'Test Accuracy (%)'})
ax.set_title('Test Accuracy (%) per Method per Fold', fontsize=16)
ax.set_ylabel('Method')
ax.set_xlabel('Fold')
plt.tight_layout()
plt.savefig(PLOTS_DIR / '01_accuracy_heatmap.png', dpi=300, bbox_inches='tight')
plt.close()
print("Saved: 01_accuracy_heatmap.png")


Saved: 01_accuracy_heatmap.png


In [16]:
# ============================================================================
# PLOT 2: Box plot of test accuracy across folds for each method
# ============================================================================
fig, ax = plt.subplots(figsize=(16, 7))
methods_sorted = df_summary.sort_values('Mean Test Acc (%)', ascending=False)['Method'].tolist()
order = methods_sorted

sns.boxplot(data=df_all, x='Method', y='Test Accuracy (%)', order=order, ax=ax, palette='Set2')
sns.stripplot(data=df_all, x='Method', y='Test Accuracy (%)', order=order, ax=ax, 
              color='black', alpha=0.5, size=4, jitter=True)
ax.set_title('Test Accuracy Distribution Across Folds', fontsize=16)
ax.set_xlabel('')
plt.xticks(rotation=60, ha='right')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS_DIR / '02_accuracy_boxplot.png', dpi=300, bbox_inches='tight')
plt.close()
print("Saved: 02_accuracy_boxplot.png")


Saved: 02_accuracy_boxplot.png


In [17]:
# ============================================================================
# PLOT 3: Feature Retention per method (bar chart with error bars)
# ============================================================================
fig, ax = plt.subplots(figsize=(16, 7))
feat_summary = df_all.groupby('Method')['Feature Retention (%)'].agg(['mean', 'std']).reindex(ALL_METHODS)
bars = ax.bar(feat_summary.index, feat_summary['mean'], yerr=feat_summary['std'], capsize=3, alpha=0.7, color='coral')
ax.axhline(y=100, color='r', linestyle='--', label='All Features (100%)')
ax.set_ylabel('Feature Retention (%)', fontsize=12)
ax.set_title('Feature Retention by Method', fontsize=16)
ax.legend()
plt.xticks(rotation=60, ha='right')
ax.grid(axis='y', alpha=0.3)

# Value labels
for bar, mean_val in zip(bars, feat_summary['mean']):
    if not np.isnan(mean_val):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 2,
                f'{mean_val:.1f}%', ha='center', va='bottom', fontsize=8, fontweight='bold')

plt.tight_layout()
plt.savefig(PLOTS_DIR / '03_feature_retention.png', dpi=300, bbox_inches='tight')
plt.close()
print("Saved: 03_feature_retention.png")


Saved: 03_feature_retention.png


In [18]:
# ============================================================================
# PLOT 4: Accuracy vs Feature Retention trade-off (scatter)
# ============================================================================
fig, ax = plt.subplots(figsize=(12, 8))

for method in ALL_METHODS:
    mdf = df_all[df_all['Method'] == method]
    if len(mdf) == 0:
        continue
    ax.scatter(mdf['Feature Retention (%)'], mdf['Test Accuracy (%)'], 
               label=method, s=60, alpha=0.7)
    # Draw mean point larger
    ax.scatter(mdf['Feature Retention (%)'].mean(), mdf['Test Accuracy (%)'].mean(),
               s=150, marker='X', edgecolors='black', linewidths=1.5)

ax.set_xlabel('Feature Retention (%)', fontsize=12)
ax.set_ylabel('Test Accuracy (%)', fontsize=12)
ax.set_title('Accuracy vs Feature Retention Trade-off', fontsize=16)
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS_DIR / '04_accuracy_vs_retention.png', dpi=300, bbox_inches='tight')
plt.close()
print("Saved: 04_accuracy_vs_retention.png")


Saved: 04_accuracy_vs_retention.png


In [19]:
# ============================================================================
# PLOT 5: Per-modality retention heatmap (average across folds)
# ============================================================================
modality_data = {}
for method in ALL_METHODS:
    modality_data[method] = {}
    for mod in FEATURE_DIMS.keys():
        col = f'{mod}_retention (%)'
        if col in df_all.columns:
            mdf = df_all[df_all['Method'] == method]
            modality_data[method][mod] = mdf[col].mean() if len(mdf) > 0 else 0

mod_df = pd.DataFrame(modality_data).T
mod_df = mod_df.reindex(ALL_METHODS)

fig, ax = plt.subplots(figsize=(10, 10))
sns.heatmap(mod_df, annot=True, fmt='.1f', cmap='RdYlGn', ax=ax, linewidths=0.5,
            vmin=0, vmax=100, cbar_kws={'label': 'Retention (%)'})
ax.set_title('Average Modality-wise Feature Retention (%)', fontsize=14)
ax.set_ylabel('Method')
ax.set_xlabel('Modality')
plt.tight_layout()
plt.savefig(PLOTS_DIR / '05_modality_retention_heatmap.png', dpi=300, bbox_inches='tight')
plt.close()
print("Saved: 05_modality_retention_heatmap.png")


Saved: 05_modality_retention_heatmap.png


In [20]:
# ============================================================================
# PLOT 6: Execution time comparison
# ============================================================================
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# FS time
time_summary = df_all.groupby('Method')['FS Time (s)'].agg(['mean', 'std']).reindex(ALL_METHODS)
bars = axes[0].bar(time_summary.index, time_summary['mean'], yerr=time_summary['std'], capsize=3, alpha=0.7, color='steelblue')
axes[0].set_ylabel('Feature Selection Time (s)', fontsize=12)
axes[0].set_title('Feature Selection Time', fontsize=14)
plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=60, ha='right')
axes[0].grid(axis='y', alpha=0.3)

# Total time
total_summary = df_all.groupby('Method')['Total Time (s)'].agg(['mean', 'std']).reindex(ALL_METHODS)
bars2 = axes[1].bar(total_summary.index, total_summary['mean'], yerr=total_summary['std'], capsize=3, alpha=0.7, color='darkorange')
axes[1].set_ylabel('Total Time (s)', fontsize=12)
axes[1].set_title('Total Time (FS + Training)', fontsize=14)
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=60, ha='right')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(PLOTS_DIR / '06_execution_time.png', dpi=300, bbox_inches='tight')
plt.close()
print("Saved: 06_execution_time.png")


Saved: 06_execution_time.png


In [21]:
# ============================================================================
# PLOT 7: Model size & dataset size comparison
# ============================================================================
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Model params
param_summary = df_all.groupby('Method')['Model Params'].mean().reindex(ALL_METHODS)
axes[0].bar(param_summary.index, param_summary.values, alpha=0.7, color='mediumpurple')
axes[0].set_ylabel('Model Parameters', fontsize=12)
axes[0].set_title('Average Model Parameters', fontsize=14)
plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=60, ha='right')
axes[0].grid(axis='y', alpha=0.3)

# Dataset reduction
ds_summary = df_all.groupby('Method')['Dataset Reduction (%)'].mean().reindex(ALL_METHODS)
axes[1].bar(ds_summary.index, ds_summary.values, alpha=0.7, color='seagreen')
axes[1].set_ylabel('Dataset Size Reduction (%)', fontsize=12)
axes[1].set_title('Average Dataset Size Reduction', fontsize=14)
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=60, ha='right')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(PLOTS_DIR / '07_model_dataset_size.png', dpi=300, bbox_inches='tight')
plt.close()
print("Saved: 07_model_dataset_size.png")


Saved: 07_model_dataset_size.png


In [22]:
# ============================================================================
# PLOT 8: Metaheuristics-only comparison (grouped bar: accuracy & retention)
# ============================================================================
meta_methods = [m for m in ALL_METHODS if m.startswith('meta_')]

if len(meta_methods) > 0:
    meta_df = df_all[df_all['Method'].isin(meta_methods)]
    
    fig, axes = plt.subplots(2, 1, figsize=(16, 12))
    
    # Accuracy
    meta_acc = meta_df.groupby('Method')['Test Accuracy (%)'].agg(['mean', 'std']).reindex(meta_methods)
    bars = axes[0].bar(range(len(meta_methods)), meta_acc['mean'], yerr=meta_acc['std'], 
                        capsize=4, alpha=0.7, color=plt.cm.tab20(np.linspace(0, 1, len(meta_methods))))
    axes[0].set_xticks(range(len(meta_methods)))
    axes[0].set_xticklabels([m.replace('meta_', '') for m in meta_methods], rotation=45, ha='right')
    axes[0].set_ylabel('Test Accuracy (%)')
    axes[0].set_title('Metaheuristics: Test Accuracy Comparison', fontsize=14)
    axes[0].grid(axis='y', alpha=0.3)
    
    for i, (bar, val) in enumerate(zip(bars, meta_acc['mean'])):
        if not np.isnan(val):
            axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + meta_acc['std'].iloc[i] + 0.5,
                        f'{val:.1f}', ha='center', va='bottom', fontsize=8)
    
    # Feature retention
    meta_feat = meta_df.groupby('Method')['Feature Retention (%)'].agg(['mean', 'std']).reindex(meta_methods)
    bars2 = axes[1].bar(range(len(meta_methods)), meta_feat['mean'], yerr=meta_feat['std'],
                         capsize=4, alpha=0.7, color=plt.cm.tab20(np.linspace(0, 1, len(meta_methods))))
    axes[1].set_xticks(range(len(meta_methods)))
    axes[1].set_xticklabels([m.replace('meta_', '') for m in meta_methods], rotation=45, ha='right')
    axes[1].set_ylabel('Feature Retention (%)')
    axes[1].set_title('Metaheuristics: Feature Retention Comparison', fontsize=14)
    axes[1].grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / '08_metaheuristics_comparison.png', dpi=300, bbox_inches='tight')
    plt.close()
    print("Saved: 08_metaheuristics_comparison.png")


Saved: 08_metaheuristics_comparison.png


In [23]:
# ============================================================================
# PLOT 9: Convergence curves for metaheuristics (average across folds)
# ============================================================================
meta_methods = [m for m in ALL_METHODS if m.startswith('meta_')]

if len(meta_methods) > 0:
    fig, ax = plt.subplots(figsize=(14, 8))
    
    for method in meta_methods:
        all_conv = []
        for fold_idx in range(N_FOLDS):
            if fold_idx in master_results[method]:
                conv = master_results[method][fold_idx].get('convergence', None)
                if conv is not None:
                    all_conv.append(conv)
        
        if all_conv:
            # Pad to same length
            max_len = max(len(c) for c in all_conv)
            padded = []
            for c in all_conv:
                if len(c) < max_len:
                    c = list(c) + [c[-1]] * (max_len - len(c))
                padded.append(c)
            
            avg_conv = np.mean(padded, axis=0)
            ax.plot(avg_conv, label=method.replace('meta_', ''), linewidth=1.5)
    
    ax.set_xlabel('Iteration', fontsize=12)
    ax.set_ylabel('Fitness (1 - Accuracy)', fontsize=12)
    ax.set_title('Metaheuristic Convergence Curves (averaged across folds)', fontsize=14)
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / '09_convergence_curves.png', dpi=300, bbox_inches='tight')
    plt.close()
    print("Saved: 09_convergence_curves.png")


Saved: 09_convergence_curves.png


In [24]:
# ============================================================================
# PLOT 10: Feature importance - which features most commonly retained/removed
# ============================================================================
# Count how many times each feature index was selected across all methods and folds
feature_freq = np.zeros(TOTAL_FEATURES)
feature_freq_per_method = {}

for method in ALL_METHODS:
    if method == 'baseline':
        continue
    method_freq = np.zeros(TOTAL_FEATURES)
    count = 0
    for fold_idx in range(N_FOLDS):
        if fold_idx in master_results[method]:
            mask = master_results[method][fold_idx]['feature_mask']
            feature_freq += mask.astype(float)
            method_freq += mask.astype(float)
            count += 1
    if count > 0:
        feature_freq_per_method[method] = method_freq / count

# Normalize overall
total_runs = (len(ALL_METHODS) - 1) * N_FOLDS  # exclude baseline
feature_freq_normalized = feature_freq / max(total_runs, 1)

# Plot feature selection frequency
fig, axes = plt.subplots(2, 1, figsize=(18, 10))

# Overall
axes[0].bar(range(TOTAL_FEATURES), feature_freq_normalized, width=1.0, alpha=0.7, color='steelblue')
axes[0].set_xlabel('Feature Index')
axes[0].set_ylabel('Selection Frequency')
axes[0].set_title('Feature Selection Frequency Across All Methods and Folds', fontsize=14)

# Add modality boundaries
start = 0
colors = ['red', 'green', 'blue', 'purple']
for i, (mod, dim) in enumerate(FEATURE_DIMS.items()):
    axes[0].axvline(x=start, color=colors[i], linestyle='--', alpha=0.5, label=f'{mod} start')
    start += dim
axes[0].legend(fontsize=8)
axes[0].grid(axis='y', alpha=0.3)

# Top 20 most and least selected features
top_20_selected = np.argsort(feature_freq_normalized)[::-1][:20]
top_20_removed = np.argsort(feature_freq_normalized)[:20]

x_pos = np.arange(20)
width = 0.35
axes[1].bar(x_pos - width/2, feature_freq_normalized[top_20_selected], width, label='Top 20 Most Selected', color='green', alpha=0.7)
axes[1].bar(x_pos + width/2, feature_freq_normalized[top_20_removed], width, label='Top 20 Least Selected', color='red', alpha=0.7)
axes[1].set_xlabel('Rank')
axes[1].set_ylabel('Selection Frequency')
axes[1].set_title('Most vs Least Frequently Selected Features', fontsize=14)
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(PLOTS_DIR / '10_feature_frequency.png', dpi=300, bbox_inches='tight')
plt.close()
print("Saved: 10_feature_frequency.png")

# Save feature frequency data
freq_df = pd.DataFrame({
    'feature_index': range(TOTAL_FEATURES),
    'overall_selection_freq': feature_freq_normalized,
})
# Add modality labels
start = 0
modality_labels = []
for mod, dim in FEATURE_DIMS.items():
    modality_labels.extend([mod] * dim)
freq_df['modality'] = modality_labels
freq_df.to_csv(PLOTS_DIR / 'feature_selection_frequency.csv', index=False)
print("Saved: feature_selection_frequency.csv")


Saved: 10_feature_frequency.png
Saved: feature_selection_frequency.csv


In [25]:
# ============================================================================
# PLOT 11: Radar/spider chart comparing methods on multiple metrics
# ============================================================================
from matplotlib.patches import FancyBboxPatch
import matplotlib.patches as mpatches

# Select key metrics for radar
radar_metrics = ['Mean Test Acc (%)', 'Mean F1 Macro (%)', 'Mean Feature Retention (%)', 'Mean Total Time (s)']

# Normalize to [0, 1] for radar
radar_data = df_summary[['Method'] + radar_metrics].copy()
for col in radar_metrics:
    min_val = radar_data[col].min()
    max_val = radar_data[col].max()
    if max_val > min_val:
        radar_data[col] = (radar_data[col] - min_val) / (max_val - min_val)
    else:
        radar_data[col] = 0.5

# For time, invert (lower is better)
radar_data['Mean Total Time (s)'] = 1 - radar_data['Mean Total Time (s)']
radar_data = radar_data.rename(columns={'Mean Total Time (s)': 'Speed (inv. time)'})
radar_cols = ['Mean Test Acc (%)', 'Mean F1 Macro (%)', 'Mean Feature Retention (%)', 'Speed (inv. time)']

# Select subset for readability (standard + top 5 metaheuristics)
top_meta = df_summary[df_summary['Method'].str.startswith('meta_')].nlargest(5, 'Mean Test Acc (%)')['Method'].tolist()
methods_to_plot = ['baseline', 'mutual_info', 'rfe', 'lasso'] + top_meta

fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))
angles = np.linspace(0, 2 * np.pi, len(radar_cols), endpoint=False).tolist()
angles += angles[:1]

for method in methods_to_plot:
    row = radar_data[radar_data['Method'] == method]
    if len(row) == 0:
        continue
    values = row[radar_cols].values.flatten().tolist()
    values += values[:1]
    ax.plot(angles, values, linewidth=1.5, label=method)
    ax.fill(angles, values, alpha=0.05)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(radar_cols, fontsize=9)
ax.set_title('Multi-Metric Comparison (Normalized)', fontsize=14, pad=20)
ax.legend(bbox_to_anchor=(1.3, 1.1), fontsize=8)
plt.tight_layout()
plt.savefig(PLOTS_DIR / '11_radar_comparison.png', dpi=300, bbox_inches='tight')
plt.close()
print("Saved: 11_radar_comparison.png")


Saved: 11_radar_comparison.png


In [26]:
# ============================================================================
# PLOT 12: Standard vs Metaheuristics grouped comparison
# ============================================================================
standard_methods = ['baseline', 'mutual_info', 'rfe', 'lasso']
meta_methods = [m for m in ALL_METHODS if m.startswith('meta_')]

fig, axes = plt.subplots(1, 3, figsize=(20, 7))

# Group averages
std_accs = df_all[df_all['Method'].isin(standard_methods)].groupby('Method')['Test Accuracy (%)'].mean()
meta_accs = df_all[df_all['Method'].isin(meta_methods)].groupby('Method')['Test Accuracy (%)'].mean()

# Accuracy: standard vs meta
all_std = df_all[df_all['Method'].isin(standard_methods)]['Test Accuracy (%)']
all_meta = df_all[df_all['Method'].isin(meta_methods)]['Test Accuracy (%)']
bp = axes[0].boxplot([all_std, all_meta], labels=['Standard\n(MI, RFE, LASSO)', 'Metaheuristics\n(14 optimizers)'],
                      patch_artist=True)
bp['boxes'][0].set_facecolor('lightblue')
bp['boxes'][1].set_facecolor('lightsalmon')
axes[0].set_ylabel('Test Accuracy (%)')
axes[0].set_title('Standard vs Metaheuristic Accuracy', fontsize=13)
axes[0].grid(axis='y', alpha=0.3)

# Feature retention
all_std_f = df_all[(df_all['Method'].isin(standard_methods)) & (df_all['Method'] != 'baseline')]['Feature Retention (%)']
all_meta_f = df_all[df_all['Method'].isin(meta_methods)]['Feature Retention (%)']
bp2 = axes[1].boxplot([all_std_f, all_meta_f], labels=['Standard', 'Metaheuristics'],
                       patch_artist=True)
bp2['boxes'][0].set_facecolor('lightblue')
bp2['boxes'][1].set_facecolor('lightsalmon')
axes[1].set_ylabel('Feature Retention (%)')
axes[1].set_title('Standard vs Metaheuristic Feature Retention', fontsize=13)
axes[1].grid(axis='y', alpha=0.3)

# Time
all_std_t = df_all[(df_all['Method'].isin(standard_methods)) & (df_all['Method'] != 'baseline')]['Total Time (s)']
all_meta_t = df_all[df_all['Method'].isin(meta_methods)]['Total Time (s)']
bp3 = axes[2].boxplot([all_std_t, all_meta_t], labels=['Standard', 'Metaheuristics'],
                       patch_artist=True)
bp3['boxes'][0].set_facecolor('lightblue')
bp3['boxes'][1].set_facecolor('lightsalmon')
axes[2].set_ylabel('Total Time (s)')
axes[2].set_title('Standard vs Metaheuristic Time', fontsize=13)
axes[2].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(PLOTS_DIR / '12_standard_vs_metaheuristics.png', dpi=300, bbox_inches='tight')
plt.close()
print("Saved: 12_standard_vs_metaheuristics.png")


Saved: 12_standard_vs_metaheuristics.png


In [27]:
# ============================================================================
# PLOT 13: GPU memory usage comparison
# ============================================================================
fig, ax = plt.subplots(figsize=(16, 7))
gpu_summary = df_all.groupby('Method')['GPU Peak (MB)'].agg(['mean', 'std']).reindex(ALL_METHODS)
bars = ax.bar(gpu_summary.index, gpu_summary['mean'], yerr=gpu_summary['std'], capsize=3, alpha=0.7, color='gold')
ax.set_ylabel('GPU Peak Memory (MB)', fontsize=12)
ax.set_title('GPU Peak Memory Usage by Method', fontsize=16)
plt.xticks(rotation=60, ha='right')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS_DIR / '13_gpu_memory.png', dpi=300, bbox_inches='tight')
plt.close()
print("Saved: 13_gpu_memory.png")


Saved: 13_gpu_memory.png


In [28]:
# ============================================================================
# PLOT 14: Per-fold accuracy line plot (all methods)
# ============================================================================
fig, ax = plt.subplots(figsize=(16, 8))

for method in ALL_METHODS:
    mdf = df_all[df_all['Method'] == method].sort_values('Fold')
    if len(mdf) > 0:
        linewidth = 2.5 if method in ['baseline', 'mutual_info', 'rfe', 'lasso'] else 1.0
        alpha = 1.0 if method in ['baseline', 'mutual_info', 'rfe', 'lasso'] else 0.6
        ax.plot(mdf['Fold'], mdf['Test Accuracy (%)'], marker='o', markersize=4,
                label=method, linewidth=linewidth, alpha=alpha)

ax.set_xlabel('Fold', fontsize=12)
ax.set_ylabel('Test Accuracy (%)', fontsize=12)
ax.set_title('Test Accuracy Across Folds (All Methods)', fontsize=16)
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
ax.grid(alpha=0.3)
ax.set_xticks(range(1, N_FOLDS + 1))
plt.tight_layout()
plt.savefig(PLOTS_DIR / '14_per_fold_accuracy_lines.png', dpi=300, bbox_inches='tight')
plt.close()
print("Saved: 14_per_fold_accuracy_lines.png")


Saved: 14_per_fold_accuracy_lines.png


## Statistical Significance Tests

In [29]:
from scipy.stats import ttest_rel, wilcoxon

print("\n" + "="*80)
print("STATISTICAL SIGNIFICANCE TESTS (each method vs baseline)")
print("="*80)

baseline_accs = []
for fold_idx in range(N_FOLDS):
    if fold_idx in master_results['baseline']:
        baseline_accs.append(master_results['baseline'][fold_idx]['test_acc'] * 100)

stat_rows = []

for method in ALL_METHODS:
    if method == 'baseline':
        continue
    
    method_accs = []
    for fold_idx in range(N_FOLDS):
        if fold_idx in master_results[method]:
            method_accs.append(master_results[method][fold_idx]['test_acc'] * 100)
    
    if len(method_accs) != len(baseline_accs):
        continue
    
    try:
        t_stat, p_value_t = ttest_rel(method_accs, baseline_accs)
    except:
        t_stat, p_value_t = 0, 1
    
    try:
        w_stat, p_value_w = wilcoxon(method_accs, baseline_accs)
    except:
        w_stat, p_value_w = 0, 1
    
    direction = "better" if np.mean(method_accs) > np.mean(baseline_accs) else "worse"
    significant = "Yes" if p_value_t < 0.05 else "No"
    
    stat_rows.append({
        'Method': method,
        'Mean Acc (%)': np.mean(method_accs),
        'Baseline Acc (%)': np.mean(baseline_accs),
        'Diff (%)': np.mean(method_accs) - np.mean(baseline_accs),
        't-stat': t_stat,
        'p-value (t-test)': p_value_t,
        'W-stat': w_stat,
        'p-value (Wilcoxon)': p_value_w,
        'Direction': direction,
        'Significant (p<0.05)': significant,
    })
    
    print(f"\n{method.upper()} vs BASELINE:")
    print(f"  Method: {np.mean(method_accs):.2f}% vs Baseline: {np.mean(baseline_accs):.2f}%")
    print(f"  Paired t-test: t={t_stat:.4f}, p={p_value_t:.4f}")
    print(f"  Wilcoxon test: W={w_stat:.4f}, p={p_value_w:.4f}")
    print(f"  -> {direction}, {'Significant' if significant == 'Yes' else 'Not significant'}")

stat_df = pd.DataFrame(stat_rows).round(4)
stat_df.to_csv(RESULTS_ROOT / "statistical_tests.csv", index=False)
print(f"\nSaved: statistical_tests.csv")
print("\n" + "="*80)



STATISTICAL SIGNIFICANCE TESTS (each method vs baseline)

MUTUAL_INFO vs BASELINE:
  Method: 93.85% vs Baseline: 94.32%
  Paired t-test: t=-0.3109, p=0.7650
  Wilcoxon test: W=8.5000, p=0.7500
  -> worse, Not significant

RFE vs BASELINE:
  Method: 93.28% vs Baseline: 94.32%
  Paired t-test: t=-0.9007, p=0.3977
  Wilcoxon test: W=8.5000, p=0.7188
  -> worse, Not significant

LASSO vs BASELINE:
  Method: 83.17% vs Baseline: 94.32%
  Paired t-test: t=-11.1789, p=0.0000
  Wilcoxon test: W=0.0000, p=0.0078
  -> worse, Significant

META_BAT vs BASELINE:
  Method: 95.01% vs Baseline: 94.32%
  Paired t-test: t=0.4259, p=0.6830
  Wilcoxon test: W=11.0000, p=0.6562
  -> better, Not significant

META_CS vs BASELINE:
  Method: 92.70% vs Baseline: 94.32%
  Paired t-test: t=-1.5523, p=0.1645
  Wilcoxon test: W=4.0000, p=0.2188
  -> worse, Not significant

META_DE vs BASELINE:
  Method: 93.39% vs Baseline: 94.32%
  Paired t-test: t=-0.6301, p=0.5486
  Wilcoxon test: W=5.5000, p=0.3438
  -> worse, N

## Final Summary

In [30]:
print("\n" + "="*80)
print("EXPERIMENT COMPLETE - RESULTS SUMMARY")
print("="*80)

print(f"\nOutput directory: {RESULTS_ROOT}")
print(f"\nPer-method folders:")
for method in ALL_METHODS:
    method_dir = RESULTS_ROOT / method
    n_files = len(list(method_dir.glob('*')))
    print(f"  {method_dir}/ ({n_files} files)")

print(f"\nPlots directory: {PLOTS_DIR}")
for p in sorted(PLOTS_DIR.glob('*.png')):
    print(f"  {p.name}")

print(f"\nCSV files:")
for p in sorted(RESULTS_ROOT.glob('*.csv')):
    print(f"  {p.name}")

print(f"\n{'='*80}")
print("KEY FILES:")
print(f"  - all_results_per_fold.csv : Every metric for every fold x method")
print(f"  - summary_all_methods.csv  : Aggregated summary")
print(f"  - statistical_tests.csv    : Significance tests vs baseline")
print(f"  - Per-method folders       : Individual fold JSONs + feature masks")
print(f"  - plots/                   : 14 comprehensive visualizations")
print(f"{'='*80}")

# Print top 5 methods by mean accuracy
print("\nTOP 5 METHODS BY MEAN TEST ACCURACY:")
top5 = df_summary.nlargest(5, 'Mean Test Acc (%)')
for i, row in top5.iterrows():
    print(f"  {row['Method']:20s}: {row['Mean Test Acc (%)']:.2f}% ± {row['Std Test Acc (%)']:.2f}% "
          f"(Features: {row['Mean Feature Retention (%)']:.1f}%)")



EXPERIMENT COMPLETE - RESULTS SUMMARY

Output directory: results_new_no_video

Per-method folders:
  results_new_no_video/baseline/ (16 files)
  results_new_no_video/mutual_info/ (16 files)
  results_new_no_video/rfe/ (16 files)
  results_new_no_video/lasso/ (16 files)
  results_new_no_video/meta_BAT/ (16 files)
  results_new_no_video/meta_CS/ (16 files)
  results_new_no_video/meta_DE/ (16 files)
  results_new_no_video/meta_FFA/ (16 files)
  results_new_no_video/meta_GA/ (16 files)
  results_new_no_video/meta_GWO/ (16 files)
  results_new_no_video/meta_HHO/ (16 files)
  results_new_no_video/meta_JAYA/ (16 files)
  results_new_no_video/meta_MFO/ (16 files)
  results_new_no_video/meta_MVO/ (16 files)
  results_new_no_video/meta_PSO/ (16 files)
  results_new_no_video/meta_SCA/ (16 files)
  results_new_no_video/meta_SSA/ (16 files)
  results_new_no_video/meta_WOA/ (16 files)

Plots directory: results_new_no_video/plots
  01_accuracy_heatmap.png
  02_accuracy_boxplot.png
  03_feature_reten

In [31]:
# ============================================================================
# BEST METAHEURISTIC PER FOLD: Test Accuracy & Feature Retention
# ============================================================================

# Identify metaheuristic-only methods
meta_methods = [m for m in ALL_METHODS if m.startswith('meta_')]
modality_names = list(FEATURE_DIMS.keys())  # ['depth', 'sensor', 'skeleton', 'video']

print("=" * 90)
print("BEST METAHEURISTIC PER FOLD (selected by highest test accuracy)")
print("=" * 90)

best_per_fold = []  # collect per-fold best results

for fold_idx in range(N_FOLDS):
    best_method = None
    best_test_acc = -1.0
    best_result = None

    for method_name in meta_methods:
        if fold_idx in master_results[method_name]:
            r = master_results[method_name][fold_idx]
            if r['test_acc'] > best_test_acc:
                best_test_acc = r['test_acc']
                best_method = method_name
                best_result = r

    if best_result is None:
        print(f"  Fold {fold_idx + 1}: No metaheuristic results found!")
        continue

    # Extract modality-wise retention
    mod_ret = best_result['modality_retention']
    mod_retention_pct = {}
    for mod in modality_names:
        if mod in mod_ret and isinstance(mod_ret[mod], dict):
            mod_retention_pct[mod] = mod_ret[mod].get('percentage', 0.0)
        else:
            mod_retention_pct[mod] = float(mod_ret.get(mod, 0.0))

    fold_entry = {
        'Fold': fold_idx + 1,
        'Best Metaheuristic': best_method,
        'Test Accuracy (%)': best_result['test_acc'] * 100,
        'Feature Retention (%)': best_result['feature_retention_pct'],
        'Features Selected': best_result['num_features_selected'],
        'Features Total': best_result['num_features_total'],
    }
    for mod in modality_names:
        fold_entry[f'{mod} Retention (%)'] = mod_retention_pct[mod]

    best_per_fold.append(fold_entry)

    # Print per-fold info
    print(f"\n  Fold {fold_idx + 1}: Best = {best_method}")
    print(f"    Test Accuracy:      {fold_entry['Test Accuracy (%)']:.2f}%")
    print(f"    Feature Retention:  {fold_entry['Feature Retention (%)']:.2f}% "
          f"({fold_entry['Features Selected']}/{fold_entry['Features Total']})")
    mod_str = ', '.join([f"{mod}={mod_retention_pct[mod]:.1f}%" for mod in modality_names])
    print(f"    Modality Retention: {mod_str}")

# Build DataFrame
df_best_meta = pd.DataFrame(best_per_fold)

# ============================================================================
# AVERAGE ACROSS FOLDS
# ============================================================================
print("\n" + "=" * 90)
print("AVERAGE ACROSS ALL FOLDS (Best Metaheuristic per Fold)")
print("=" * 90)

avg_test_acc = df_best_meta['Test Accuracy (%)'].mean()
std_test_acc = df_best_meta['Test Accuracy (%)'].std()
avg_feat_ret = df_best_meta['Feature Retention (%)'].mean()
std_feat_ret = df_best_meta['Feature Retention (%)'].std()

print(f"\n  Average Test Accuracy:     {avg_test_acc:.2f}% ± {std_test_acc:.2f}%")
print(f"  Average Feature Retention: {avg_feat_ret:.2f}% ± {std_feat_ret:.2f}%")

print(f"\n  Average Modality-wise Feature Retention:")
for mod in modality_names:
    col = f'{mod} Retention (%)'
    avg_mod = df_best_meta[col].mean()
    std_mod = df_best_meta[col].std()
    print(f"    {mod:>10s}: {avg_mod:.2f}% ± {std_mod:.2f}%")

# ============================================================================
# SUMMARY TABLE
# ============================================================================
print("\n" + "=" * 90)
print("PER-FOLD SUMMARY TABLE")
print("=" * 90)
display_cols = ['Fold', 'Best Metaheuristic', 'Test Accuracy (%)', 'Feature Retention (%)']
display_cols += [f'{mod} Retention (%)' for mod in modality_names]
print(df_best_meta[display_cols].to_string(index=False, float_format='%.2f'))

# Save to CSV
csv_path = RESULTS_ROOT / "best_metaheuristic_per_fold.csv"
df_best_meta.to_csv(csv_path, index=False)
print(f"\nSaved to: {csv_path}")

BEST METAHEURISTIC PER FOLD (selected by highest test accuracy)

  Fold 1: Best = meta_GWO
    Test Accuracy:      97.22%
    Feature Retention:  26.30% (661/2513)
    Modality Retention: depth=28.2%, sensor=25.5%, skeleton=26.4%

  Fold 2: Best = meta_MFO
    Test Accuracy:      92.59%
    Feature Retention:  47.47% (1193/2513)
    Modality Retention: depth=48.6%, sensor=45.9%, skeleton=48.0%

  Fold 3: Best = meta_SSA
    Test Accuracy:      92.59%
    Feature Retention:  46.60% (1171/2513)
    Modality Retention: depth=44.0%, sensor=46.8%, skeleton=46.9%

  Fold 4: Best = meta_BAT
    Test Accuracy:      95.37%
    Feature Retention:  46.44% (1167/2513)
    Modality Retention: depth=39.8%, sensor=49.8%, skeleton=46.0%

  Fold 5: Best = meta_GWO
    Test Accuracy:      98.15%
    Feature Retention:  25.51% (641/2513)
    Modality Retention: depth=25.9%, sensor=24.8%, skeleton=25.7%

  Fold 6: Best = meta_PSO
    Test Accuracy:      98.13%
    Feature Retention:  46.04% (1157/2513)
  